# 🔍 Log Reliability Auditor — Point & Click

Before any process-mining project starts, the first question is always the same:
**can we actually trust this log?** This notebook runs the same automated audit a data
scientist would run — mandatory-field checks, data-quality checks, freshness checks — but
**you will not write or edit any Python code.** Every piece of analysis happens by
clicking buttons; the code cells exist so the computer can run things.

### How to use this notebook
1. Click **Runtime → Run all** (Google Colab) or **Cell → Run All** (Jupyter), top to bottom, once.
   The first cell installs a couple of packages — takes under a minute.
2. Scroll to **Step 1** and either upload a log of your own (CSV/Excel/XES), or click one of
   the two sample buttons to see the audit in action immediately — one sample is a **clean**
   log, the other is a **messy** one, so you can compare what "good" and "bad" look like side by side.
3. Scroll to **Step 2** and click through the six audit dimensions, in any order, as many
   times as you like.
4. There's a plain-language glossary of the scoring rubric near the bottom.

> 💡 **Tip:** the easiest zero-install way to run this is **[Google Colab](https://colab.research.google.com/)**
> — upload this `.ipynb` there and click Run all. Nothing to install on your own laptop.

---


## ⚙️ Setup (run once)

Installs the libraries this notebook needs. Lighter than a process-discovery notebook —
this is all tables and text, no diagram-drawing engine required.

In [1]:
# Run this cell first — takes under a minute, usually.
import sys, subprocess
def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
for _pkg in ["pandas", "openpyxl", "ipywidgets"]:
    _pip_install(_pkg)
print("✅ Setup complete — you can now run the cells below.")

✅ Setup complete — you can now run the cells below.


In [2]:
# Imports used throughout the notebook.
from __future__ import annotations
import io
import json as _json
import re
import warnings
import tempfile
import xml.etree.ElementTree as ET
from datetime import datetime, date
from pathlib import Path
from typing import Any

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

warnings.filterwarnings("ignore")
pd.set_option("mode.chained_assignment", None)

# All state for the workshop lives here, so every button's click-handler can
# read/write it without scattering global variables everywhere.
state: dict[str, Any] = {"df": None, "schema": None, "source_meta": None, "as_of": None}

print("Ready.")

Ready.


---
### Under the hood: the audit engine (run this cell, no need to read it)

This is the same engine documented in the companion `pm_log_audit.py` script / Claude
Skill — ingestion, column auto-detection, mandatory-field validation, attribute discovery,
data-quality checks, freshness assessment, and scoring. It's reproduced here so the
notebook is a single self-contained file.

In [3]:
# ---- Ingestion: reads CSV / TSV / XLSX / XLS / JSON / Parquet / XES -------
XES_NS = {"xes": "http://www.xes-standard.org/"}

def load_log(path: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Load a log file of any supported format into a flat event table."""
    suffix = path.suffix.lower()
    meta: dict[str, Any] = {"source_path": str(path), "source_format": suffix.lstrip(".")}

    if suffix in (".csv", ".tsv", ".txt"):
        sep = "\t" if suffix == ".tsv" else None
        df = pd.read_csv(path, sep=sep, engine="python", dtype=str, keep_default_na=True)
        meta["delimiter"] = "tab" if suffix == ".tsv" else "auto-detected"
    elif suffix in (".xlsx", ".xls"):
        df = pd.read_excel(path, dtype=str)
    elif suffix == ".json":
        raw = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(raw, dict):
            # try common container keys
            for key in ("events", "data", "log", "records"):
                if key in raw and isinstance(raw[key], list):
                    raw = raw[key]
                    break
        df = pd.json_normalize(raw)
        df = df.astype(str).where(df.notna(), None)
    elif suffix == ".xes":
        df, xes_meta = _load_xes(path)
        meta.update(xes_meta)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
        df = df.astype(str).where(df.notna(), None)
    else:
        raise ValueError(f"Unsupported file format: {suffix or '(none)'}")

    meta["raw_row_count"] = len(df)
    meta["raw_column_count"] = df.shape[1]
    return df, meta


def _load_xes(path: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Minimal XES parser (no pm4py dependency). Flattens trace+event attrs."""
    tree = ET.parse(path)
    root = tree.getroot()

    def strip_ns(tag: str) -> str:
        return tag.split("}")[-1]

    rows = []
    trace_count = 0
    for trace in root:
        if strip_ns(trace.tag) != "trace":
            continue
        trace_count += 1
        trace_attrs = {}
        for el in trace:
            tag = strip_ns(el.tag)
            if tag == "event":
                continue
            key = el.attrib.get("key")
            val = el.attrib.get("value")
            if key:
                trace_attrs[f"trace:{key}"] = val
        for event in trace:
            if strip_ns(event.tag) != "event":
                continue
            row = dict(trace_attrs)
            for attr in event:
                key = attr.attrib.get("key")
                val = attr.attrib.get("value")
                if key:
                    row[key] = val
            rows.append(row)

    df = pd.DataFrame(rows)
    return df, {"xes_trace_count": trace_count}




In [4]:
# ---- Schema discovery: guesses which column is which ---------------------
CASE_PATTERNS = [r"case.?id", r"^case$", r"trace.?id", r"instance.?id", r"process.?id",
                  r"^trace.?concept.?name$"]
ACTIVITY_PATTERNS = [r"activit", r"^task$", r"^event.?name$", r"concept.?name", r"^step$", r"^action$"]
TIME_PATTERNS = [r"time.?stamp", r"^time$", r"^date$", r"datetime", r"start.?time", r"end.?time",
                  r"complete.?time", r"occur"]
RESOURCE_PATTERNS = [r"resource", r"performer", r"assignee", r"^user$", r"actor", r"^agent$",
                      r"employee", r"^owner$", r"^role$"]
DEPARTMENT_PATTERNS = [r"depart", r"^team$", r"cost.?cent", r"business.?unit", r"^org(anization)?"
                        r"(al)?.?unit$", r"^site$", r"^location$", r"^division$"]
NONHUMAN_PATTERNS = [r"machine", r"system", r"application", r"equipment", r"queue", r"workstation",
                      r"^bot$", r"robot", r"^device$", r"asset"]
CASE_ATTR_PATTERNS = [r"customer", r"product", r"amount", r"priority", r"channel", r"^type$",
                       r"^category$"]
EVENT_META_PATTERNS = [r"lifecycle", r"transition", r"^cost$", r"^duration$", r"transaction.?type"]

_PATTERN_GROUPS = {
    "case_id": CASE_PATTERNS,
    "activity": ACTIVITY_PATTERNS,
    "timestamp": TIME_PATTERNS,
    "resource": RESOURCE_PATTERNS,
    "department": DEPARTMENT_PATTERNS,
    "non_human_resource": NONHUMAN_PATTERNS,
    "case_attribute": CASE_ATTR_PATTERNS,
    "event_metadata": EVENT_META_PATTERNS,
}


def _name_score(col: str, patterns: list[str]) -> float:
    norm = re.sub(r"[\s_\-]+", ".", col.strip().lower())
    for pat in patterns:
        if re.search(pat, norm):
            return 1.0
    return 0.0


def _parse_dt(series: pd.Series) -> pd.Series:
    """Parse to datetime and normalize to timezone-naive (mixed tz-aware /
    tz-naive values otherwise break comparisons and groupby operations)."""
    parsed = pd.to_datetime(series, errors="coerce", utc=True)
    try:
        parsed = parsed.dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass
    return parsed


def _is_datetime_parsable(series: pd.Series, sample_size: int = 200) -> float:
    sample = series.dropna().astype(str).head(sample_size)
    if sample.empty:
        return 0.0
    parsed = pd.to_datetime(sample, errors="coerce", utc=False)
    return float(parsed.notna().mean())


def discover_schema(df: pd.DataFrame) -> dict[str, Any]:
    """Score every column against every semantic role; pick best candidates."""
    n = len(df)
    col_stats = {}
    for col in df.columns:
        s = df[col]
        nonnull = s.notna().sum()
        uniqueness = s.nunique(dropna=True) / nonnull if nonnull else 0.0
        col_stats[col] = {
            "nonnull_rate": nonnull / n if n else 0.0,
            "uniqueness_ratio": uniqueness,
            "n_unique": int(s.nunique(dropna=True)),
        }

    scores: dict[str, dict[str, float]] = {role: {} for role in _PATTERN_GROUPS}

    for col in df.columns:
        for role, patterns in _PATTERN_GROUPS.items():
            name_sc = _name_score(col, patterns)
            heuristic_sc = 0.0
            if role == "timestamp":
                heuristic_sc = _is_datetime_parsable(df[col]) * 0.6
            elif role == "case_id":
                u = col_stats[col]["uniqueness_ratio"]
                # good case-id: repeated across events but not globally unique,
                # and not the activity/timestamp column itself
                if 0 < u < 0.95:
                    heuristic_sc = 0.3 * (1 - abs(u - 0.3))
            elif role == "activity":
                u = col_stats[col]["uniqueness_ratio"]
                if 0 < u < 0.5:
                    heuristic_sc = 0.2
            combined = min(1.0, name_sc + heuristic_sc)
            if combined > 0:
                scores[role][col] = round(combined, 3)

    mapping = {}
    confidence = {}
    for role in ("case_id", "activity", "timestamp"):
        candidates = scores[role]
        if candidates:
            best_col = max(candidates, key=candidates.get)
            mapping[role] = best_col
            confidence[role] = candidates[best_col]
        else:
            mapping[role] = None
            confidence[role] = 0.0

    # avoid mandatory fields colliding on the same column
    used = set()
    for role in ("timestamp", "case_id", "activity"):  # priority order
        col = mapping[role]
        if col in used:
            candidates = {c: v for c, v in scores[role].items() if c not in used}
            if candidates:
                best_col = max(candidates, key=candidates.get)
                mapping[role] = best_col
                confidence[role] = candidates[best_col]
            else:
                mapping[role] = None
                confidence[role] = 0.0
        if mapping[role]:
            used.add(mapping[role])

    optional_mapping = {}
    for role in ("resource", "department", "non_human_resource", "case_attribute", "event_metadata"):
        cols = [c for c, v in scores[role].items() if v >= 0.9 and c not in mapping.values()]
        optional_mapping[role] = sorted(cols, key=lambda c: -scores[role][c])

    return {
        "mandatory_mapping": mapping,
        "mandatory_confidence": confidence,
        "optional_mapping": optional_mapping,
        "role_scores": scores,
        "column_stats": col_stats,
    }



In [5]:
# ---- Validation, attribute discovery, data quality, freshness, scoring ---
CONFIDENCE_LOW_THRESHOLD = 0.6


def validate_mandatory_fields(df: pd.DataFrame, schema: dict) -> dict[str, Any]:
    mapping = schema["mandatory_mapping"]
    confidence = schema["mandatory_confidence"]
    n = len(df)
    results = {}

    # --- Case ID ---
    col = mapping.get("case_id")
    if not col:
        results["case_id"] = {"status": "MISSING", "column": None, "issues": ["No column resembling a Case ID was found."]}
    else:
        s = df[col]
        nonnull_rate = s.notna().mean()
        n_unique = s.nunique(dropna=True)
        issues = []
        if nonnull_rate < 1.0:
            issues.append(f"{(1 - nonnull_rate) * 100:.1f}% of rows have a missing case id.")
        if n_unique == n:
            issues.append("Every case id is unique across all rows — no case appears to repeat across multiple events; check this is really an event log and not a summary table.")
        if n_unique == 1:
            issues.append("Only a single distinct case id value exists.")
        status = "OK" if not issues and confidence.get("case_id", 0) >= CONFIDENCE_LOW_THRESHOLD else (
            "WARNING" if not issues else "WARNING")
        if confidence.get("case_id", 0) < CONFIDENCE_LOW_THRESHOLD:
            issues.append(f"Column matched by weak heuristics only (confidence {confidence.get('case_id',0):.2f}); please confirm manually.")
        results["case_id"] = {
            "status": status if issues else "OK",
            "column": col,
            "confidence": confidence.get("case_id", 0),
            "nonnull_rate": round(float(nonnull_rate), 4),
            "distinct_cases": int(n_unique),
            "issues": issues,
        }

    # --- Activity ---
    col = mapping.get("activity")
    if not col:
        results["activity"] = {"status": "MISSING", "column": None, "issues": ["No column resembling an Activity label was found."]}
    else:
        s = df[col].dropna().astype(str)
        nonnull_rate = df[col].notna().mean()
        distinct = s.nunique()
        issues = []
        if nonnull_rate < 1.0:
            issues.append(f"{(1 - nonnull_rate) * 100:.1f}% of rows have a missing activity label.")
        # near-duplicate labels via fuzzy matching (casing/whitespace variants)
        norm = s.str.strip().str.lower()
        variants: dict[str, set[str]] = {}
        for orig, nrm in zip(s.unique(), pd.Series(s.unique()).str.strip().str.lower()):
            variants.setdefault(nrm, set()).add(orig)
        dup_groups = {k: v for k, v in variants.items() if len(v) > 1}
        if dup_groups:
            examples = "; ".join(f"{sorted(v)}" for v in list(dup_groups.values())[:5])
            issues.append(f"{len(dup_groups)} activity label(s) appear as casing/whitespace variants of each other, e.g. {examples}.")
        if distinct > max(50, n * 0.5):
            issues.append(f"{distinct} distinct activity labels for {n} events — unusually high; check this isn't a free-text field or leaking case-specific IDs.")
        if distinct <= 1:
            issues.append("Only one distinct activity value — log has no discriminative process steps.")
        results["activity"] = {
            "status": "OK" if not issues else "WARNING",
            "column": col,
            "confidence": confidence.get("activity", 0),
            "nonnull_rate": round(float(nonnull_rate), 4),
            "distinct_activities": int(distinct),
            "issues": issues,
        }

    # --- Timestamp ---
    col = mapping.get("timestamp")
    if not col:
        results["timestamp"] = {"status": "MISSING", "column": None, "issues": ["No column resembling a Timestamp was found."]}
    else:
        raw = df[col]
        parsed = _parse_dt(raw)
        nonnull_rate = raw.notna().mean()
        parse_fail_rate = (raw.notna() & parsed.isna()).mean()
        issues = []
        if nonnull_rate < 1.0:
            issues.append(f"{(1 - nonnull_rate) * 100:.1f}% of rows have a missing timestamp.")
        if parse_fail_rate > 0:
            issues.append(f"{parse_fail_rate * 100:.1f}% of non-missing values could not be parsed as a date/time.")
        # granularity check: date-only vs full datetime
        time_components = parsed.dropna().dt.time
        date_only = (time_components == datetime.min.time()).mean() if len(time_components) else 0
        if date_only > 0.95:
            issues.append("Timestamps appear to be date-only (no time-of-day component) — this limits activity ordering within a day.")
        # monotonicity within case (only if case_id resolved)
        case_col = mapping.get("case_id")
        order_violations = 0
        if case_col:
            tmp = pd.DataFrame({"case": df[case_col], "ts": parsed}).dropna()
            if not tmp.empty:
                viol = tmp.groupby("case")["ts"].apply(lambda x: (x.diff().dt.total_seconds() < 0).sum())
                order_violations = int(viol.sum())
            if order_violations > 0:
                issues.append(f"{order_violations} event(s) are out of chronological order within their case — the log may not be pre-sorted, or timestamps are unreliable.")
        # future dates
        now = pd.Timestamp.now()
        future = (parsed > now).sum()
        if future > 0:
            issues.append(f"{int(future)} event(s) have a timestamp in the future relative to today.")
        results["timestamp"] = {
            "status": "OK" if not issues else "WARNING",
            "column": col,
            "confidence": confidence.get("timestamp", 0),
            "nonnull_rate": round(float(nonnull_rate), 4),
            "parse_failure_rate": round(float(parse_fail_rate), 4),
            "date_only": bool(date_only > 0.95),
            "order_violations": order_violations,
            "future_events": int(future),
            "issues": issues,
        }

    any_missing = any(r["status"] == "MISSING" for r in results.values())
    results["_overall"] = "CRITICAL" if any_missing else (
        "WARNING" if any(r["status"] == "WARNING" for r in results.values()) else "OK")
    return results


# --------------------------------------------------------------------------
# 3. ATTRIBUTE DISCOVERY
# --------------------------------------------------------------------------

ATTR_LABELS = {
    "resource": "Human resource utilization (who performed each event)",
    "department": "Organizational context (department / team / business unit)",
    "non_human_resource": "Non-human resource utilization (machines/systems/equipment)",
    "case_attribute": "Case-level business attributes (customer, product, amount, priority...)",
    "event_metadata": "Event-level metadata (lifecycle transition, cost, duration...)",
}


def discover_attributes(df: pd.DataFrame, schema: dict) -> dict[str, Any]:
    n = len(df)
    out = {}
    for role, label in ATTR_LABELS.items():
        cols = schema["optional_mapping"].get(role, [])
        entries = []
        for col in cols:
            s = df[col]
            fill_rate = s.notna().mean()
            entries.append({
                "column": col,
                "fill_rate": round(float(fill_rate), 4),
                "distinct_values": int(s.nunique(dropna=True)),
            })
        out[role] = {
            "label": label,
            "present": len(entries) > 0,
            "columns": entries,
        }
    return out


# --------------------------------------------------------------------------
# 4. DATA QUALITY & ANOMALY DETECTION
# --------------------------------------------------------------------------

def data_quality_checks(df: pd.DataFrame, schema: dict) -> dict[str, Any]:
    n = len(df)
    mapping = schema["mandatory_mapping"]
    issues: list[dict[str, Any]] = []

    # missingness per column
    completeness = {
        col: round(float(df[col].notna().mean()), 4) for col in df.columns
    }
    sparse_cols = [c for c, r in completeness.items() if r < 0.5]
    if sparse_cols:
        issues.append({
            "severity": "INFO",
            "check": "sparse_columns",
            "detail": f"{len(sparse_cols)} column(s) are less than 50% filled: {sparse_cols[:10]}{'...' if len(sparse_cols) > 10 else ''}",
        })

    # exact duplicate rows
    dup_rows = int(df.duplicated().sum())
    if dup_rows:
        issues.append({
            "severity": "WARNING",
            "check": "duplicate_rows",
            "detail": f"{dup_rows} fully duplicate row(s) found ({dup_rows / n * 100:.2f}% of events).",
        })

    # duplicate (case, activity, timestamp) triples
    case_col, act_col, time_col = mapping.get("case_id"), mapping.get("activity"), mapping.get("timestamp")
    dup_triples = 0
    if case_col and act_col and time_col:
        key_cols = [case_col, act_col, time_col]
        dup_triples = int(df.duplicated(subset=key_cols).sum())
        if dup_triples:
            issues.append({
                "severity": "WARNING",
                "check": "duplicate_case_activity_timestamp",
                "detail": f"{dup_triples} row(s) share the same case id, activity and timestamp — likely duplicate log entries needing a tiebreaker.",
            })

    # case id reappearance after a long gap (possible ID reuse)
    reused_cases = 0
    if case_col and time_col:
        parsed = _parse_dt(df[time_col])
        tmp = pd.DataFrame({"case": df[case_col], "ts": parsed}).dropna()
        if not tmp.empty:
            spans = tmp.groupby("case")["ts"].agg(["min", "max", "count"])
            overall_span = (parsed.max() - parsed.min()).total_seconds() if parsed.notna().any() else 0
            if overall_span > 0:
                long_gap_ratio_threshold = 0.5
                gaps = tmp.sort_values("ts").groupby("case")["ts"].apply(
                    lambda x: x.diff().dt.total_seconds().max() if len(x) > 1 else 0)
                reused_cases = int((gaps > overall_span * long_gap_ratio_threshold).sum())
        if reused_cases:
            issues.append({
                "severity": "INFO",
                "check": "possible_case_id_reuse",
                "detail": f"{reused_cases} case id(s) have an internal gap spanning more than half the log's total time range — verify case ids aren't being recycled.",
            })

    # outlier case durations
    outlier_cases = 0
    if case_col and time_col:
        parsed = _parse_dt(df[time_col])
        tmp = pd.DataFrame({"case": df[case_col], "ts": parsed}).dropna()
        if not tmp.empty:
            durations = tmp.groupby("case")["ts"].agg(lambda x: (x.max() - x.min()).total_seconds())
            if len(durations) > 4:
                q1, q3 = durations.quantile(0.25), durations.quantile(0.75)
                iqr = q3 - q1
                if iqr > 0:
                    upper = q3 + 3 * iqr
                    outlier_cases = int((durations > upper).sum())
        if outlier_cases:
            issues.append({
                "severity": "INFO",
                "check": "case_duration_outliers",
                "detail": f"{outlier_cases} case(s) have a duration more than 3xIQR above the norm — check for stuck/erroneous cases before duration-based analysis.",
            })

    # mixed-type / inconsistent formatting flags on non-mapped columns
    inconsistent_cols = []
    for col in df.columns:
        if col in (case_col, act_col, time_col):
            continue
        s = df[col].dropna().astype(str)
        if s.empty:
            continue
        looks_numeric = s.str.match(r"^-?\d+(\.\d+)?$")
        if 0 < looks_numeric.mean() < 1:
            inconsistent_cols.append(col)
    if inconsistent_cols:
        issues.append({
            "severity": "INFO",
            "check": "mixed_type_columns",
            "detail": f"{len(inconsistent_cols)} column(s) mix numeric-looking and non-numeric text values: {inconsistent_cols[:10]}.",
        })

    return {
        "completeness_by_column": completeness,
        "duplicate_rows": dup_rows,
        "duplicate_case_activity_timestamp": dup_triples,
        "possible_case_id_reuse": reused_cases,
        "case_duration_outliers": outlier_cases,
        "issues": issues,
    }


# --------------------------------------------------------------------------
# 5. FRESHNESS / AGE ASSESSMENT
# --------------------------------------------------------------------------

def freshness_assessment(df: pd.DataFrame, schema: dict, as_of: pd.Timestamp) -> dict[str, Any]:
    time_col = schema["mandatory_mapping"].get("timestamp")
    if not time_col:
        return {"status": "UNKNOWN", "detail": "No timestamp column resolved; cannot assess freshness."}

    parsed = _parse_dt(df[time_col]).dropna()
    if parsed.empty:
        return {"status": "UNKNOWN", "detail": "No parsable timestamps found."}

    # exclude future-dated events (relative to as_of) from freshness math — these
    # are almost always data errors and would otherwise mask genuine staleness.
    future_mask = parsed > as_of
    n_future_excluded = int(future_mask.sum())
    valid = parsed[~future_mask]
    if valid.empty:
        valid = parsed  # fall back if literally everything is "future"

    first, last = valid.min(), valid.max()
    span_days = (last - first).total_seconds() / 86400
    staleness_days = (as_of - last).total_seconds() / 86400

    # coverage gap analysis: bucket by week, find empty weeks within the span
    weekly_counts = valid.dt.to_period("W").value_counts().sort_index()
    full_range = pd.period_range(weekly_counts.index.min(), weekly_counts.index.max(), freq="W")
    missing_weeks = int(len(full_range) - len(weekly_counts))
    gap_ratio = missing_weeks / len(full_range) if len(full_range) else 0

    staleness_flag = "STALE" if staleness_days > 180 else ("AGING" if staleness_days > 60 else "FRESH")

    return {
        "status": staleness_flag,
        "first_event": first.isoformat(),
        "last_event": last.isoformat(),
        "span_days": round(span_days, 1),
        "days_since_last_event": round(staleness_days, 1),
        "weeks_with_no_events": missing_weeks,
        "weeks_with_no_events_ratio": round(gap_ratio, 3),
        "as_of": as_of.isoformat(),
        "future_events_excluded": n_future_excluded,
    }


# --------------------------------------------------------------------------
# 6. STATISTICAL PROFILE
# --------------------------------------------------------------------------

def statistical_profile(df: pd.DataFrame, schema: dict) -> dict[str, Any]:
    mapping = schema["mandatory_mapping"]
    case_col, act_col, time_col = mapping.get("case_id"), mapping.get("activity"), mapping.get("timestamp")
    profile: dict[str, Any] = {"n_events": len(df)}

    if case_col:
        profile["n_cases"] = int(df[case_col].nunique(dropna=True))
        events_per_case = df.groupby(case_col).size()
        profile["avg_events_per_case"] = round(float(events_per_case.mean()), 2)
        profile["min_events_per_case"] = int(events_per_case.min())
        profile["max_events_per_case"] = int(events_per_case.max())

    if act_col:
        profile["n_distinct_activities"] = int(df[act_col].nunique(dropna=True))

    if case_col and act_col:
        variants = df.groupby(case_col)[act_col].apply(lambda x: tuple(x))
        profile["n_variants"] = int(variants.nunique())

    if case_col and time_col:
        parsed = _parse_dt(df[time_col])
        tmp = pd.DataFrame({"case": df[case_col], "ts": parsed}).dropna()
        if not tmp.empty:
            durations_hours = tmp.groupby("case")["ts"].agg(
                lambda x: (x.max() - x.min()).total_seconds() / 3600)
            profile["avg_case_duration_hours"] = round(float(durations_hours.mean()), 2)
            profile["median_case_duration_hours"] = round(float(durations_hours.median()), 2)

    resource_cols = schema["optional_mapping"].get("resource", [])
    if resource_cols:
        profile["n_resources"] = int(df[resource_cols[0]].nunique(dropna=True))

    return profile


# --------------------------------------------------------------------------
# 7. SCORING & REPORT GENERATION
# --------------------------------------------------------------------------

WEIGHTS = {"mandatory": 0.40, "quality": 0.25, "attributes": 0.20, "freshness": 0.15}
GRADE_BANDS = [(90, "Excellent"), (75, "Good"), (55, "Fair"), (30, "Poor"), (0, "Unusable")]


def _grade(score: float) -> str:
    for threshold, label in GRADE_BANDS:
        if score >= threshold:
            return label
    return "Unusable"


def score_log(mandatory: dict, quality: dict, attributes: dict, freshness: dict, n_events: int) -> dict[str, Any]:
    # mandatory score
    mand_score = 0.0
    for role in ("case_id", "activity", "timestamp"):
        r = mandatory.get(role, {})
        if r.get("status") == "OK":
            mand_score += 100 / 3
        elif r.get("status") == "WARNING":
            mand_score += (100 / 3) * max(0.3, 1 - 0.15 * len(r.get("issues", [])))
        # MISSING contributes 0

    # quality score: penalize by issue severity, normalized by log size
    quality_score = 100.0
    severity_penalty = {"WARNING": 15, "INFO": 5}
    for issue in quality.get("issues", []):
        quality_score -= severity_penalty.get(issue["severity"], 5)
    quality_score = max(0.0, quality_score)

    # attribute richness score
    n_present = sum(1 for v in attributes.values() if v["present"])
    attr_score = (n_present / len(attributes)) * 100 if attributes else 0.0

    # freshness score
    freshness_map = {"FRESH": 100, "AGING": 60, "STALE": 20, "UNKNOWN": 30}
    fresh_score = freshness_map.get(freshness.get("status", "UNKNOWN"), 30)
    gap_ratio = freshness.get("weeks_with_no_events_ratio", 0) or 0
    fresh_score = max(0.0, fresh_score - gap_ratio * 30)

    overall = (
        mand_score * WEIGHTS["mandatory"]
        + quality_score * WEIGHTS["quality"]
        + attr_score * WEIGHTS["attributes"]
        + fresh_score * WEIGHTS["freshness"]
    )

    if mandatory.get("_overall") == "CRITICAL":
        overall = min(overall, 25.0)  # cap: unusable without mandatory fields

    return {
        "mandatory_score": round(mand_score, 1),
        "quality_score": round(quality_score, 1),
        "attribute_score": round(attr_score, 1),
        "freshness_score": round(fresh_score, 1),
        "overall_score": round(overall, 1),
        "grade": _grade(overall),
    }


def build_recommendations(mandatory, attributes, quality, freshness) -> list[str]:
    recs = []
    for role, label in (("case_id", "Case ID"), ("activity", "Activity"), ("timestamp", "Timestamp")):
        r = mandatory.get(role, {})
        if r.get("status") == "MISSING":
            recs.append(f"CRITICAL: Add/identify a {label} column — process mining cannot proceed without it.")
        for issue in r.get("issues", []):
            recs.append(f"[{label}] {issue}")

    for role, info in attributes.items():
        if not info["present"]:
            recs.append(f"Consider adding '{info['label']}' — not currently available, limiting related analyses (e.g. handover-of-work, organizational mining, resource bottlenecks).")

    for issue in quality.get("issues", []):
        if issue["severity"] in ("WARNING",):
            recs.append(f"[Data quality] {issue['detail']}")

    if freshness.get("status") == "STALE":
        recs.append(f"Log is stale — last event was {freshness.get('days_since_last_event', '?'):.0f} days ago. Confirm this reflects the current process before drawing operational conclusions.")
    if freshness.get("weeks_with_no_events_ratio", 0) > 0.15:
        recs.append("Notable gaps in event coverage over time — investigate whether the extraction missed periods of activity.")

    return recs



---
## 📥 Step 1 — Load a Log to Audit

Upload your own export, or click a sample button to see the audit run immediately.

In [6]:
# ---- Two built-in sample logs: one clean, one messy ----------------------
# Both are embedded directly so the sample buttons always work, with no
# internet access or extra files needed.

_CLEAN_CSV = """Case ID,Activity,Timestamp,Resource,Department
REQ-0001,Submit Request,2026-08-03 09:02,Priya,IT Support
REQ-0001,Triage Request,2026-08-03 09:40,Omar,IT Support
REQ-0001,Resolve Request,2026-08-03 14:10,Jun,IT Support
REQ-0001,Close Request,2026-08-04 09:00,Priya,IT Support
REQ-0002,Submit Request,2026-08-03 10:15,Alice,IT Support
REQ-0002,Triage Request,2026-08-03 10:50,Omar,IT Support
REQ-0002,Resolve Request,2026-08-03 15:30,Jun,IT Support
REQ-0002,Close Request,2026-08-04 09:10,Alice,IT Support
REQ-0003,Submit Request,2026-08-04 08:45,Priya,IT Support
REQ-0003,Triage Request,2026-08-04 09:20,Omar,IT Support
REQ-0003,Escalate Request,2026-08-04 11:00,Omar,IT Support
REQ-0003,Resolve Request,2026-08-05 10:00,Jun,IT Support
REQ-0003,Close Request,2026-08-05 15:00,Priya,IT Support
REQ-0004,Submit Request,2026-08-05 09:05,Alice,IT Support
REQ-0004,Triage Request,2026-08-05 09:45,Omar,IT Support
REQ-0004,Resolve Request,2026-08-05 13:00,Jun,IT Support
REQ-0004,Close Request,2026-08-06 09:00,Alice,IT Support
REQ-0005,Submit Request,2026-08-06 09:00,Priya,IT Support
REQ-0005,Triage Request,2026-08-06 09:30,Omar,IT Support
REQ-0005,Resolve Request,2026-08-06 14:45,Jun,IT Support
REQ-0005,Close Request,2026-08-07 09:00,Priya,IT Support
REQ-0006,Submit Request,2026-08-07 08:50,Alice,IT Support
REQ-0006,Triage Request,2026-08-07 09:25,Omar,IT Support
REQ-0006,Escalate Request,2026-08-07 11:30,Omar,IT Support
REQ-0006,Resolve Request,2026-08-08 10:15,Jun,IT Support
REQ-0006,Close Request,2026-08-08 16:00,Alice,IT Support
REQ-0007,Submit Request,2026-08-08 09:00,Priya,IT Support
REQ-0007,Triage Request,2026-08-08 09:40,Omar,IT Support
REQ-0007,Resolve Request,2026-08-08 13:30,Jun,IT Support
REQ-0007,Close Request,2026-08-09 09:00,Priya,IT Support
REQ-0008,Submit Request,2026-08-09 09:10,Alice,IT Support
REQ-0008,Triage Request,2026-08-09 09:50,Omar,IT Support
REQ-0008,Resolve Request,2026-08-09 14:00,Jun,IT Support
REQ-0008,Close Request,2026-08-10 09:00,Alice,IT Support
"""

_MESSY_CSV = """case_id,activity,ts,handler
ORD-001,Register Order,2023-01-02 09:00,alice
ORD-001,Check Stock,2023-01-02 10:15,bob
ORD-001,approve payment ,2023-01-02 14:30,
ORD-001,Ship Order,2023-01-03 08:00,dave
ORD-001,Close Order,2023-01-05 09:00,alice
ORD-002,Register Order,2023-01-02 09:20,alice
ORD-002,Check Stock,2023-01-02 11:00,bob
ORD-002,Approve Payment,2023-01-06 15:00,
ORD-002,Ship Order,2023-01-07 08:30,dave
ORD-002,Close Order,2023-01-09 09:00,alice
,Register Order,2023-01-03 09:10,eve
ORD-003,Check Stock,2023-01-03 09:45,bob
ORD-003,Approve Payment,2023-01-03 10:30,carol
ORD-003,Ship Order,2023-01-03 16:00,dave
ORD-003,Close Order,2023-01-04 09:00,eve
ORD-004,Register Order,2023-01-04 09:00,alice
ORD-004,Check Stock,2023-01-04 09:40,bob
ORD-004,Approve Payment,2023-01-04 10:00,
ORD-004,Approve Payment,2023-01-04 10:00,
ORD-004,Ship Order,2023-01-05 15:00,dave
ORD-004,Close Order,2023-01-06 09:00,alice
ORD-005,Register Order,2023-01-05 09:05,eve
ORD-005,Check Stock,2023-01-05 09:30,bob
ORD-005,Approve Payment,2023-01-05 11:00,carol
ORD-005,Ship Order,2099-01-05 17:00,dave
ORD-005,Close Order,2023-01-07 09:00,eve
ORD-006,Register Order,2023-01-08 09:00,alice
ORD-006,Check Stock,2023-01-08 09:50,bob
ORD-006,Approve Payment,2023-01-08 13:00,carol
ORD-006,Ship Order,2023-01-09 08:00,dave
ORD-006,Close Order,2023-01-10 09:00,alice
ORD-007,register order,2023-04-09 09:00,eve
ORD-007,Check Stock,2023-01-09 09:30,bob
ORD-007,Approve Payment,2023-01-14 11:00,carol
ORD-007,Ship Order,2023-01-14 17:00,dave
ORD-007,Close Order,2023-01-16 09:00,eve
"""

def get_clean_df():
    return pd.read_csv(io.StringIO(_CLEAN_CSV))

def get_messy_df():
    return pd.read_csv(io.StringIO(_MESSY_CSV))

print("Sample datasets ready: one clean, one messy.")

Sample datasets ready: one clean, one messy.


In [7]:
# ---- Step 1 widgets: upload / sample buttons / column confirmation ------
upload_widget = widgets.FileUpload(accept=".csv,.tsv,.xlsx,.xls,.json,.xes", multiple=False,
                                    description="Upload log")
clean_button = widgets.Button(description="Try the CLEAN sample", button_style="success", icon="check")
messy_button = widgets.Button(description="Try the MESSY sample", button_style="danger", icon="warning")
step1_output = widgets.Output()

def _read_uploaded_file(change):
    """Handles both ipywidgets v7 and v8 FileUpload.value shapes. Writes the
    upload to a temp file and reuses the exact same load_log() used by the
    standalone script/skill, so behavior is identical either way."""
    new_val = change["new"]
    if not new_val:
        return None, None
    if isinstance(new_val, dict):  # ipywidgets v7: {filename: {...}}
        fname = list(new_val.keys())[0]
        content = new_val[fname]["content"]
    else:  # ipywidgets v8: tuple of file-info mappings
        info = new_val[0]
        fname = info["name"]
        content = info["content"]
    content_bytes = bytes(content)
    suffix = Path(fname).suffix or ".csv"
    tmp = Path(tempfile.gettempdir()) / f"_uploaded_log{suffix}"
    tmp.write_bytes(content_bytes)
    df, meta = load_log(tmp)
    meta["source_path"] = fname  # show the real filename, not the temp path
    return df, meta

def _df_to_meta(df, name, fmt):
    return {"source_path": name, "source_format": fmt,
            "raw_row_count": len(df), "raw_column_count": df.shape[1]}

def render_column_confirmation(df, source_meta):
    schema = discover_schema(df)
    guesses = schema["mandatory_mapping"]
    confidence = schema["mandatory_confidence"]
    cols = list(df.columns)

    def _label(role, name):
        c = confidence.get(role, 0)
        flag = " ⚠️ low confidence" if c < CONFIDENCE_LOW_THRESHOLD else ""
        return f"{name} (confidence {c:.2f}{flag}):"

    case_dd = widgets.Dropdown(options=cols, value=guesses.get("case_id") or cols[0],
                                description=_label("case_id", "Case ID"), style={"description_width": "260px"})
    act_dd = widgets.Dropdown(options=cols, value=guesses.get("activity") or cols[0],
                               description=_label("activity", "Activity"), style={"description_width": "260px"})
    time_dd = widgets.Dropdown(options=cols, value=guesses.get("timestamp") or cols[0],
                                description=_label("timestamp", "Timestamp"), style={"description_width": "260px"})
    asof_picker = widgets.DatePicker(description="Audit as of:", value=date.today())
    confirm_btn = widgets.Button(description="✅ Confirm & run schema discovery", button_style="success")
    confirm_out = widgets.Output()

    def on_confirm(b):
        with confirm_out:
            clear_output()
            try:
                overrides = {"case_id": case_dd.value, "activity": act_dd.value, "timestamp": time_dd.value}
                final_schema = discover_schema(df)  # re-run fresh, then apply user's confirmed choices
                for role, col in overrides.items():
                    final_schema["mandatory_mapping"][role] = col
                    final_schema["mandatory_confidence"][role] = 1.0  # user-confirmed

                state["df"] = df
                state["schema"] = final_schema
                state["source_meta"] = source_meta
                state["as_of"] = pd.Timestamp(asof_picker.value)

                n_cases = df[case_dd.value].nunique(dropna=True)
                opt_found = [v["label"] for v in discover_attributes(df, final_schema).values() if v["present"]]
                msg = f"""**✅ Log ready: `{source_meta['source_path']}`**

| | |
|---|---|
| Rows x Columns | {source_meta['raw_row_count']} x {source_meta['raw_column_count']} |
| Distinct cases (by chosen Case ID column) | {n_cases} |
| Optional attributes auto-detected | {', '.join(opt_found) if opt_found else 'none'} |
"""
                display(Markdown(msg))
                display(Markdown("**You're set — scroll down to Step 2 and click through the audit dimensions.**"))
            except Exception as e:
                print(f"❌ Something went wrong preparing the log: {{e}}")
                print("Double-check the column choices above and try again.")

    confirm_btn.on_click(on_confirm)
    display(Markdown(f"**Loaded `{source_meta['source_path']}`** — "
                      f"{source_meta['raw_row_count']} rows, {source_meta['raw_column_count']} columns.\n\n"
                      "Here's my best guess at which column is which (low-confidence guesses are flagged — "
                      "worth double-checking those). Adjust if needed, then confirm."))
    display(widgets.VBox([case_dd, act_dd, time_dd, asof_picker, confirm_btn, confirm_out]))

def on_upload_change(change):
    with step1_output:
        clear_output()
        try:
            df, meta = _read_uploaded_file(change)
        except Exception as e:
            print(f"❌ Couldn't read that file: {{e}}")
            return
        if df is None:
            return
        render_column_confirmation(df, meta)

def on_clean_click(b):
    with step1_output:
        clear_output()
        df = get_clean_df()
        render_column_confirmation(df, _df_to_meta(df, "clean_sample.csv (built-in)", "csv"))

def on_messy_click(b):
    with step1_output:
        clear_output()
        df = get_messy_df()
        render_column_confirmation(df, _df_to_meta(df, "messy_sample.csv (built-in)", "csv"))

upload_widget.observe(on_upload_change, names="value")
clean_button.on_click(on_clean_click)
messy_button.on_click(on_messy_click)

display(widgets.HBox([upload_widget, clean_button, messy_button]))
display(step1_output)

**Loaded `Reimbursement_Event_Log_25Cases.csv`** — 76 rows, 10 columns.

Here's my best guess at which column is which (low-confidence guesses are flagged — worth double-checking those). Adjust if needed, then confirm.

---
## 🔎 Step 2 — Run the Audit

Once Step 1 shows "Log ready", click any button below, in any order. Each one runs a
different part of the audit and explains what it found in plain language.

In [10]:
# ---- Section renderers: turn each audit function's output into Markdown --
def _log_ready():
    if state.get("df") is None:
        print("⚠️ Please load a log in Step 1 first.")
        return False
    return True

def _run_all_checks():
    df, schema, as_of = state["df"], state["schema"], state["as_of"]
    mandatory = validate_mandatory_fields(df, schema)
    attributes = discover_attributes(df, schema)
    quality = data_quality_checks(df, schema)
    freshness = freshness_assessment(df, schema, as_of)
    profile = statistical_profile(df, schema)
    score = score_log(mandatory, quality, attributes, freshness, len(df))
    recs = build_recommendations(mandatory, attributes, quality, freshness)
    return dict(mandatory=mandatory, attributes=attributes, quality=quality,
                freshness=freshness, profile=profile, score=score, recommendations=recs)

def render_mandatory_md(mandatory):
    lines = ["### 1️⃣ Mandatory Fields\n",
             "Case ID, Activity and Timestamp are non-negotiable — without all three, "
             "process mining cannot proceed.\n",
             "| Field | Status | Detected Column | Confidence | Notes |",
             "|---|---|---|---|---|"]
    for role, label in (("case_id", "Case ID"), ("activity", "Activity"), ("timestamp", "Timestamp")):
        r = mandatory.get(role, {})
        col = r.get("column") or "—"
        conf = f"{r.get('confidence', 0):.2f}" if r.get("column") else "—"
        notes = "; ".join(r.get("issues", [])) or "OK"
        lines.append(f"| {label} | {r.get('status')} | `{col}` | {conf} | {notes} |")
    if mandatory.get("_overall") == "CRITICAL":
        lines.append("\n> 🛑 **CRITICAL: this log is not usable for process mining as-is.**")
    return "\n".join(lines)

def render_attributes_md(attributes):
    lines = ["### 2️⃣ Attribute Richness\n",
             "What else is in the log besides the bare minimum? More of these means more "
             "kinds of analysis become possible (who does the work, which department, "
             "which systems are involved...).\n",
             "| Category | Present? | Columns (fill rate) |", "|---|---|---|"]
    for info in attributes.values():
        parts = [f"`{c['column']}` ({c['fill_rate']*100:.0f}%)" for c in info["columns"]]
        cols_str = ", ".join(parts) or "—"
        present_str = "Yes" if info["present"] else "No"
        lines.append(f"| {info['label']} | {present_str} | {cols_str} |")
    return "\n".join(lines)

def render_quality_md(quality):
    lines = ["### 3️⃣ Data Quality & Anomalies\n",
             f"- Duplicate rows: **{quality['duplicate_rows']}**",
             f"- Duplicate (case, activity, timestamp) rows: **{quality['duplicate_case_activity_timestamp']}**",
             f"- Possible case-id reuse (large internal gaps): **{quality['possible_case_id_reuse']}**",
             f"- Case duration outliers (>3xIQR): **{quality['case_duration_outliers']}**\n"]
    if quality["issues"]:
        lines += ["| Severity | Check | Detail |", "|---|---|---|"]
        for issue in quality["issues"]:
            lines.append(f"| {issue['severity']} | {issue['check']} | {issue['detail']} |")
    else:
        lines.append("No additional data-quality issues detected.")
    return "\n".join(lines)

def render_freshness_md(freshness):
    lines = ["### 4️⃣ Log Age & Freshness\n"]
    if freshness.get("status") in ("FRESH", "AGING", "STALE"):
        lines += [f"- Status: **{freshness['status']}**",
                  f"- First event: {freshness['first_event']}",
                  f"- Last event: {freshness['last_event']}",
                  f"- Span: {freshness['span_days']} days",
                  f"- Days since last event (as of {freshness['as_of'][:10]}): {freshness['days_since_last_event']}",
                  f"- Weeks with no recorded events: {freshness['weeks_with_no_events']} "
                  f"({freshness['weeks_with_no_events_ratio']*100:.1f}% of span)"]
        if freshness.get("future_events_excluded"):
            lines.append(f"- Note: {freshness['future_events_excluded']} future-dated event(s) were "
                          "excluded from this calculation (flagged separately under Mandatory Fields).")
    else:
        lines.append(freshness.get("detail", "Freshness could not be assessed."))
    return "\n".join(lines)

def render_profile_md(profile):
    lines = ["### 5️⃣ Statistical Profile\n"]
    for k, v in profile.items():
        lines.append(f"- **{k.replace('_', ' ')}**: {v}")
    return "\n".join(lines)

def render_verdict_md(score, recommendations):
    lines = [f"### 🎯 Overall Verdict: **{score['grade']}** ({score['overall_score']}/100)\n",
             "| Dimension | Score | Weight |", "|---|---|---|",
             f"| Mandatory field integrity | {score['mandatory_score']}/100 | 40% |",
             f"| Data completeness & anomalies | {score['quality_score']}/100 | 25% |",
             f"| Attribute richness | {score['attribute_score']}/100 | 20% |",
             f"| Freshness / recency | {score['freshness_score']}/100 | 15% |", "",
             "#### Recommendations"]
    if recommendations:
        lines += [f"- {r}" for r in recommendations]
    else:
        lines.append("- No blocking issues found. Log is ready for standard process mining workflows.")
    return "\n".join(lines)

print("Section renderers ready.")

Section renderers ready.


In [11]:
# ---- Step 2: the six audit-dimension buttons -----------------------------
explore_output = widgets.Output()

def show_mandatory(b=None):
    with explore_output:
        clear_output()
        if not _log_ready(): return
        r = _run_all_checks()
        display(Markdown(render_mandatory_md(r["mandatory"])))

def show_attributes(b=None):
    with explore_output:
        clear_output()
        if not _log_ready(): return
        r = _run_all_checks()
        display(Markdown(render_attributes_md(r["attributes"])))

def show_quality(b=None):
    with explore_output:
        clear_output()
        if not _log_ready(): return
        r = _run_all_checks()
        display(Markdown(render_quality_md(r["quality"])))

def show_freshness(b=None):
    with explore_output:
        clear_output()
        if not _log_ready(): return
        r = _run_all_checks()
        display(Markdown(render_freshness_md(r["freshness"])))

def show_profile(b=None):
    with explore_output:
        clear_output()
        if not _log_ready(): return
        r = _run_all_checks()
        display(Markdown(render_profile_md(r["profile"])))

def show_verdict(b=None):
    with explore_output:
        clear_output()
        if not _log_ready(): return
        r = _run_all_checks()
        display(Markdown(render_verdict_md(r["score"], r["recommendations"])))

def show_full_report(b=None):
    with explore_output:
        clear_output()
        if not _log_ready(): return
        r = _run_all_checks()
        display(Markdown(render_verdict_md(r["score"], r["recommendations"])))
        display(Markdown(render_mandatory_md(r["mandatory"])))
        display(Markdown(render_attributes_md(r["attributes"])))
        display(Markdown(render_quality_md(r["quality"])))
        display(Markdown(render_freshness_md(r["freshness"])))
        display(Markdown(render_profile_md(r["profile"])))

btn_mandatory = widgets.Button(description="1️⃣ Mandatory Fields", button_style="primary")
btn_attributes = widgets.Button(description="2️⃣ Attribute Richness", button_style="primary")
btn_quality = widgets.Button(description="3️⃣ Data Quality", button_style="warning")
btn_freshness = widgets.Button(description="4️⃣ Freshness", button_style="warning")
btn_profile = widgets.Button(description="5️⃣ Key Numbers")
btn_verdict = widgets.Button(description="🎯 Overall Verdict", button_style="success")
btn_full = widgets.Button(description="📋 Full Report")
btn_clear = widgets.Button(description="🧹 Clear", button_style="danger")

btn_mandatory.on_click(show_mandatory)
btn_attributes.on_click(show_attributes)
btn_quality.on_click(show_quality)
btn_freshness.on_click(show_freshness)
btn_profile.on_click(show_profile)
btn_verdict.on_click(show_verdict)
btn_full.on_click(show_full_report)
btn_clear.on_click(lambda b: explore_output.clear_output())

display(widgets.HBox([btn_mandatory, btn_attributes, btn_quality, btn_freshness, btn_profile]))
display(widgets.HBox([btn_verdict, btn_full, btn_clear]))
display(explore_output)

Output()

---
## 📖 Glossary — the scoring rubric

| Dimension | Weight | What it checks |
|---|---|---|
| Mandatory field integrity | 40% | Can we confidently identify Case ID, Activity, and Timestamp, and are they clean (no excess missing values, no out-of-order timestamps, no near-duplicate activity labels)? |
| Data completeness & anomalies | 25% | Duplicate rows, duplicate (case, activity, timestamp) entries, possible case-id reuse, outlier case durations, sparse/mixed-type columns. |
| Attribute richness | 20% | Beyond the bare minimum: is there a resource/performer field? A department/org-unit field? Non-human resource (machine/system) data? |
| Freshness / recency | 15% | How recent is the most recent event, and are there gaps in time coverage? |

**Grade bands:** Excellent 90–100 · Good 75–89 · Fair 55–74 · Poor 30–54 · Unusable 0–29
(automatically capped at 25 if any mandatory field is missing — a log without an
identifiable Case ID, Activity, or Timestamp can't support process mining no matter how
clean the rest of it is).

> Try running both sample logs through **🎯 Overall Verdict** and compare — the clean one
> should land in Excellent/Good, the messy one in Poor/Fair. That contrast is the fastest
> way to show a cohort what this kind of audit actually catches.

---
## 🚀 Where to go from here

This notebook is built for **live exploration in a workshop setting**. For ongoing use
without opening a notebook each time, the same audit engine is also available as:

- A standalone Python script (`pm_log_audit.py`) runnable from the command line.
- An installable Claude Skill, so anyone can ask an AI assistant in plain English to
  audit a log and get this same report back, with no notebook involved at all.

**To try this with your own data:** scroll back to Step 1, upload your file, confirm the
three columns, and click through Step 2 again.